In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

In [2]:
df = pd.read_pickle("../data/processed/dataset_with_target.pkl")

df.shape


(3000000, 168)

In [12]:
import numpy as np

for c in df.columns:
    if c != "flag":
        df[c] = df[c].astype(np.float32)

df.dtypes.head()

MemoryError: Unable to allocate 11.4 MiB for an array with shape (3000000,) and data type float32

In [6]:
from catboost import Pool

TARGET_COL = "flag"

# 1. Забираем таргет (это дёшево)
y = df[TARGET_COL]

# 2. УДАЛЯЕМ колонку таргета ИЗ df (in-place!)
del df[TARGET_COL]

# 3. Теперь df содержит ТОЛЬКО признаки
train_pool = Pool(
    data=df,
    label=y
)



In [9]:
model = CatBoostClassifier(
    iterations=1200,
    depth=9,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    subsample=0.8,
    rsm=0.8,
    l2_leaf_reg=5,
    eval_fraction=0.2,
    use_best_model=True,
    verbose=100
)

In [11]:
model.fit(train_pool)

CatBoostError: bad allocation

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred = model.predict_proba(train_pool)[:, 1]
roc_auc = roc_auc_score(y, y_pred)

roc_auc
